In [61]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

In [62]:
df_bing = pd.read_csv("bing_campaign_stats.csv")
df_google = pd.read_csv("google_ads_campaign_stats.csv")
df_meta = pd.read_csv("meta_ads_campaign_stats.csv")
#Reading the data frames

In [63]:
print(len(df_bing))
print(len(df_google))
print(len(df_meta))

2873
19272
3417


Expenditure Features -> (Spend , Impression , Clicks , Conversion) Raw Data
(CTR , CPC , CVR , CPA , ROAS , budget_utilization , CPM) Derived Data/ Data Metrics
(Channel , Campaign_Type = {
    Search , Shopping , Performance_Max , Display => Mapped across the three datasets 
})

NO CPA AND CVR FOR META DATASET

Time Features -> 



In [64]:
def Feature_Extraction(df_bing, df_google, df_meta):
    # =========================================================================
    # 1. BING FEATURES
    # =========================================================================
    df_bing["Channel"] = "bing"
    df_bing["CTR"] = df_bing["Clicks"] / df_bing["Impressions"].replace(0, float('nan'))
    df_bing["CPC"] = df_bing["Spend"] / df_bing["Clicks"].replace(0, float('nan'))
    df_bing["CVR"] = df_bing["Conversions"] / df_bing["Clicks"].replace(0, float('nan'))
    df_bing["CPA"] = df_bing["Spend"] / df_bing["Conversions"].replace(0, float('nan'))
    df_bing["ROAS"] = df_bing["Revenue"] / df_bing["Spend"].replace(0, float('nan'))    
    df_bing["budget_utilization"] = df_bing["Spend"] / df_bing["DailyBudget"].replace(0, float('nan'))
    df_bing["CPM"] = (df_bing["Spend"] * 1000) / df_bing["Impressions"].replace(0, float('nan'))

    bing_map = {
        'Search': 'Search',
        'Shopping': 'Shopping',
        'PerformanceMax': 'Performance_Max',
        'Audience': 'Display'
    }
    df_bing["Campaign_Type"] = df_bing["CampaignType"].map(bing_map)
    df_bing.drop(columns=["CampaignName", "CampaignType"], inplace=True)
    
    # =========================================================================
    # 2. GOOGLE FEATURES
    # =========================================================================
    df_google["Channel"] = "google"
    df_google["Spend"] = df_google["metrics_cost_micros"] / 1000000
    df_google["CTR"] = df_google["metrics_clicks"] / df_google["metrics_impressions"].replace(0, float('nan'))
    df_google["CPC"] = df_google["Spend"] / df_google["metrics_clicks"].replace(0, float('nan'))
    df_google["CVR"] = df_google["metrics_conversions"] / df_google["metrics_clicks"].replace(0, float('nan'))
    df_google["CPA"] = df_google["Spend"] / df_google["metrics_conversions"].replace(0, float('nan'))
    df_google["ROAS"] = df_google["metrics_conversions_value"] / df_google["Spend"].replace(0, float('nan'))
    df_google["budget_utilization"] = df_google["Spend"] / df_google["campaign_budget_amount"].replace(0, float('nan'))  
    df_google["CPM"] = (df_google["Spend"] * 1000) / df_google["metrics_impressions"].replace(0, float('nan'))  
    
    df_google.rename(columns={
        "metrics_clicks": "Clicks",
        "metrics_conversions": "Conversions",
        "metrics_impressions": "Impressions",
        "campaign_budget_amount": "DailyBudget",
        "campaign_id": "CampaignId",
        "segments_date": "TimePeriod",
        "metrics_conversions_value": "Revenue" # Keep revenue consistent!
    }, inplace=True)

    google_map = {
        'SEARCH': 'Search',
        'SHOPPING': 'Shopping',
        'PERFORMANCE_MAX': 'Performance_Max',
        'DISPLAY': 'Display',
        'VIDEO': 'Display',
        'DEMAND_GEN': 'Display'
    }
    df_google["Campaign_Type"] = df_google["campaign_advertising_channel_type"].map(google_map)
    df_google.drop(columns=["metrics_cost_micros", "metrics_video_views", "campaign_name", "campaign_advertising_channel_type"], inplace=True)
    
    # =========================================================================
    # 3. META FEATURES
    # =========================================================================
    df_meta["Channel"] = "meta"
    df_meta["Revenue"] = df_meta["conversion"] 
    df_meta["ROAS"] = df_meta["Revenue"] / df_meta["spend"].replace(0, float('nan'))
    
    # Meta lacks conversion counts and CPA data in this file
    df_meta["Conversions"] = np.nan
    df_meta["CPA"] = np.nan
    df_meta["CVR"] = np.nan

    def categorize_meta(name):
        name_str = str(name)
        if 'DPA' in name_str:
            return 'Shopping'
        elif 'Adv_Plus' in name_str:
            return 'Performance_Max'
        else:
            return 'Display'
            
    df_meta["Campaign_Type"] = df_meta["campaign_name"].apply(categorize_meta)

    df_meta.rename(columns={
        "campaign_id": "CampaignId",
        "date_start": "TimePeriod",
        "cpc": "CPC",
        "cpm": "CPM",
        "ctr": "CTR",
        "spend": "Spend",
        "clicks": "Clicks",
        "impressions": "Impressions",
        "daily_budget": "DailyBudget"
    }, inplace=True)

    df_meta["budget_utilization"] = df_meta["Spend"] / df_meta["DailyBudget"].replace(0, float('nan'))
    df_meta.drop(columns=["reach", "campaign_name", "conversion"], inplace=True)
    return df_bing, df_google, df_meta


In [65]:
df_bing_proc, df_google_proc, df_meta_proc = Feature_Extraction(df_bing, df_google, df_meta)
final_df = pd.concat([df_bing_proc, df_google_proc, df_meta_proc], ignore_index=True)
if "Unnamed: 0" in final_df.columns:
    final_df.drop(columns=["Unnamed: 0"], inplace=True)

zero_fill_columns = ["Spend", "DailyBudget", "Impressions", "Clicks", "Revenue"]
final_df[zero_fill_columns] = final_df[zero_fill_columns].fillna(0)

nan_columns = ["CTR", "CPC", "CPM", "CVR", "CPA", "ROAS", "budget_utilization", "Conversions"]
for col in nan_columns:
    final_df[col] = final_df[col].replace({0: np.nan}) # Ensures no accidental 0s creep into rates

final_df["TimePeriod"] = pd.to_datetime(final_df["TimePeriod"])
column_order = [
    "TimePeriod", "Channel", "CampaignId", "Campaign_Type",
    "Spend", "DailyBudget", "budget_utilization",
    "Impressions", "Clicks", "Conversions", "Revenue",
    "CTR", "CPC", "CPM", "CVR", "CPA", "ROAS"
]
final_df = final_df[column_order]

print("--- Smart Merge Complete ---")
print(f"Total Rows: {final_df.shape[0]}")
print(f"Blank budgets remaining (Should be 0): {final_df['DailyBudget'].isna().sum()}")
print(f"Blank CVRs remaining (Expected for Meta): {final_df['CVR'].isna().sum()}")

--- Smart Merge Complete ---
Total Rows: 25562
Blank budgets remaining (Should be 0): 0
Blank CVRs remaining (Expected for Meta): 12593


In [66]:
final_df.head()

,TimePeriod,Channel,CampaignId,Campaign_Type,Spend,DailyBudget,budget_utilization,Impressions,Clicks,Conversions,Revenue,CTR,CPC,CPM,CVR,CPA,ROAS
0,2024-05-25,bing,566560838,Search,4.70,10.0,0.470,140.0,22.0,NaN,0.0,0.157143,0.213636,33.571429,NaN,NaN,NaN
1,2024-05-26,bing,566560838,Search,4.30,10.0,0.430,120.0,14.0,NaN,0.0,0.116667,0.307143,35.833333,NaN,NaN,NaN
2,2024-05-27,bing,566560838,Search,4.41,10.0,0.441,149.0,18.0,NaN,0.0,0.120805,0.245000,29.597315,NaN,NaN,NaN
3,2024-05-28,bing,566560838,Search,2.61,10.0,0.261,135.0,8.0,NaN,0.0,0.059259,0.326250,19.333333,NaN,NaN,NaN
4,2024-05-29,bing,566560838,Search,2.56,10.0,0.256,145.0,10.0,NaN,0.0,0.068966,0.256000,17.655172,NaN,NaN,NaN


In [70]:
import time
from prophet import Prophet
import statsmodels.formula.api as smf

# Ensure time column is a datetime object outside the loop
final_df["TimePeriod"] = pd.to_datetime(final_df["TimePeriod"])
channels = ["bing", "google", "meta"]

def calculate_wape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / np.sum(y_true)

for channel in channels:
    df_channel = final_df[final_df["Channel"] == channel].copy()
    
    df_channel.rename(columns={
        "TimePeriod": "ds",
        "Spend": "spend",
        "Revenue": "y"
    }, inplace=True)
    
    # Aggregate daily data into clean weekly blocks
    df_macro = df_channel.resample('W-MON', on='ds').agg({
        "y": "sum",
        "spend": "sum"
    }).reset_index()
    
    df_macro = df_macro.sort_values(by="ds").reset_index(drop=True)
    
    # Chronological Train/Test Split (80/20)
    split = int(len(df_macro) * 0.80)
    df_train = df_macro.iloc[:split].copy()
    df_test = df_macro.iloc[split:].copy()
    
    y_test_actual = df_test["y"].values
    
    print(f"\n=========================================")
    print(f"TRAINING CHANNEL: {channel.upper()}")
    print(f"=========================================")
    
    start_time = time.time()
    
    if channel == "google":
        # Strategy A: High-Volume Structural Time-Series Model
        model = Prophet(
            growth='linear',
            seasonality_mode='multiplicative',
            changepoint_prior_scale=0.05,
            interval_width=0.80, 
            weekly_seasonality=False,
            daily_seasonality=False,
            yearly_seasonality=True
        )
        model.add_regressor('spend')
        model.add_country_holidays(country_name='US')
        model.fit(df_train)
        
        df_fcst = model.predict(df_test)
        y_pred_median = df_fcst["yhat"].values
        y_pred_10percentile = df_fcst["yhat_lower"].values
        
        # Extract Beta for the downstream Causal Inference Layer
        from prophet.utilities import regressor_coefficients
        coef_df = regressor_coefficients(model)
        beta_weight = coef_df[coef_df['regressor'] == 'spend']['coef'].values[0]
        print(f"Prophet Spend Coefficient (Beta): {beta_weight:.6f}")
        
    else:
        # Strategy B: Low-Volume Regularized Quantile Regression Engine
        # Engineer robust continuous seasonality features (Sine/Cosine waves) 
        # to block Patsy train/test category level alignment issues.
        for df in [df_train, df_test]:
            df['trend_idx'] = np.arange(len(df))
            df['sin_year'] = np.sin(2 * np.pi * df['ds'].dt.dayofyear / 365.25)
            df['cos_year'] = np.cos(2 * np.pi * df['ds'].dt.dayofyear / 365.25)
            
        formula = 'y ~ spend + trend_idx + sin_year + cos_year'
        
        # Fit Median Model (50th percentile)
        mod_50 = smf.quantreg(formula, df_train)
        res_50 = mod_50.fit(q=0.5, max_iter=2500)
        y_pred_median = res_50.predict(df_test).values
        
        # Fit Conservative Risk-Mitigated Model (10th percentile)
        mod_10 = smf.quantreg(formula, df_train)
        res_10 = mod_10.fit(q=0.1, max_iter=2500)
        y_pred_10percentile = res_10.predict(df_test).values
        
        # Clip negative predictions to 0 to adhere to realistic business logic
        y_pred_median = np.clip(y_pred_median, 0, None)
        y_pred_10percentile = np.clip(y_pred_10percentile, 0, None)
        
        # Extract Beta Weight directly out of the formula matrix
        beta_weight = res_50.params['spend']
        print(f"QuantReg Spend Coefficient (Beta): {beta_weight:.6f}")
        
    end_time = time.time()
    print(f"Training completed in {end_time - start_time:.2f} seconds.")
    
    # ===================================================
    # METRIC EVALUATION
    # ===================================================
    wape_median = calculate_wape(y_test_actual, y_pred_median)
    wape_10pc = calculate_wape(y_test_actual, y_pred_10percentile)
    
    print(f"ERROR (WAPE) - Median (50% Quantile): {wape_median:.4f}")
    if channel in ["meta", "bing"]:
        print(f"ERROR (WAPE) - Conservative (10% Quantile): {wape_10pc:.4f}")
        
    print(f"--- Operational Summary (Test Window) ---")
    print(f"Total Actual Revenue:        ${np.sum(y_test_actual):,.2f}")
    print(f"Total Predicted (Median):    ${np.sum(y_pred_median):,.2f}")
    if channel in ["meta", "bing"]:
        print(f"Total Predicted (10% Bound): ${np.sum(y_pred_10percentile):,.2f}")

13:55:09 - cmdstanpy - INFO - Chain [1] start processing
13:55:09 - cmdstanpy - INFO - Chain [1] done processing



TRAINING CHANNEL: BING
QuantReg Spend Coefficient (Beta): -0.844354
Training completed in 0.05 seconds.
ERROR (WAPE) - Median (50% Quantile): 2.0040
ERROR (WAPE) - Conservative (10% Quantile): 0.9981
--- Operational Summary (Test Window) ---
Total Actual Revenue:        $11,287.03
Total Predicted (Median):    $21,229.85
Total Predicted (10% Bound): $257.65

TRAINING CHANNEL: GOOGLE
Prophet Spend Coefficient (Beta): 0.000038
Training completed in 0.18 seconds.
ERROR (WAPE) - Median (50% Quantile): 0.3296
--- Operational Summary (Test Window) ---
Total Actual Revenue:        $2,323,422.09
Total Predicted (Median):    $1,881,805.67

TRAINING CHANNEL: META
QuantReg Spend Coefficient (Beta): 5.233843
Training completed in 0.05 seconds.
ERROR (WAPE) - Median (50% Quantile): 0.4631
ERROR (WAPE) - Conservative (10% Quantile): 0.5163
--- Operational Summary (Test Window) ---
Total Actual Revenue:        $244,841.73
Total Predicted (Median):    $263,076.60
Total Predicted (10% Bound): $122,508.